In [21]:
# # Connecting Google Drive
# from google.colab import drive
# drive.mount('/content/drive')

In [22]:
# # Copy dataset from GDrive
# import os
# import shutil

# # Google Drive dataset
# source_root = "/content/drive/MyDrive/DeepLearning/Plant_Leaf_Disease_Detector-CNN/dataset"

# # Local Colab destination
# destination_root = "/content/dataset"

# # Classes you want
# selected_classes = [
#     "Potato___Early_blight",
#     "Potato___healthy",
#     "Potato___Late_blight"
# ]

# # Splits
# splits = ["train", "val", "test"]

# for split in splits:
#     for class_name in selected_classes:
        
#         source = os.path.join(source_root, split, class_name)
#         destination = os.path.join(destination_root, split, class_name)

#         if os.path.exists(source):
#             shutil.copytree(source, destination, dirs_exist_ok=True)
#             print(f"Copied: {split}/{class_name}")
#         else:
#             print(f"Not found: {source}")

# print("\nDone!")

In [23]:
# Importing libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models 

In [24]:
IMAGE_SIZE = 256
BATCH_SIZE = 32
CHANNELS = 3

In [25]:
# Reading and preprocessing the dataset 
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
  '/content/dataset/train',
  shuffle=True,
  image_size=(IMAGE_SIZE, IMAGE_SIZE),
  batch_size = BATCH_SIZE
)
test_ds = tf.keras.preprocessing.image_dataset_from_directory(
  '/content/dataset/test',
  shuffle=False,
  image_size=(IMAGE_SIZE, IMAGE_SIZE),
  batch_size = BATCH_SIZE
)
val_ds = tf.keras.preprocessing.image_dataset_from_directory(
  '/content/dataset/val',
  shuffle=False,
  image_size=(IMAGE_SIZE, IMAGE_SIZE),
  batch_size = BATCH_SIZE
)

Found 1506 files belonging to 3 classes.
Found 216 files belonging to 3 classes.
Found 430 files belonging to 3 classes.


In [26]:
# Optimize dataset for fast training
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size = tf.data.AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size = tf.data.AUTOTUNE)
test_ds = test_ds.cache().prefetch(buffer_size = tf.data.AUTOTUNE)

In [32]:
# Building model
input_shape = (IMAGE_SIZE, IMAGE_SIZE, CHANNELS)
model = models.Sequential([
  
  layers.Input(shape=input_shape),

  # Rescaling, and Augementation the images
  layers.Rescaling(1.0/255),
  layers.RandomFlip('horizontal_and_vertical'),
  layers.RandomRotation(0.2),

  layers.Conv2D(32, (3,3), activation='relu'),
  layers.MaxPool2D((2,2)),

  layers.Conv2D(64, (3,3), activation='relu'),
  layers.MaxPool2D((2,2)),

  layers.Conv2D(64, (3,3), activation='relu'),
  layers.MaxPool2D((2,2)),

  layers.Conv2D(64, (3,3), activation='relu'),
  layers.MaxPool2D((2,2)),

  layers.Conv2D(64, (3,3), activation='relu'),
  layers.MaxPool2D((2,2)),

  layers.Conv2D(64, (3,3), activation='relu'),
  layers.MaxPool2D((2,2)),

  layers.Flatten(),
  
  layers.Dense(64, activation='relu'),
  layers.Dense(3, activation='softmax')
])

In [33]:
model.summary()

Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling_4 (Rescaling)         │ (None, 256, 256, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_flip_3 (RandomFlip)      │ (None, 256, 256, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_rotation_3               │ (None, 256, 256, 3)    │             0 │
│ (RandomRotation)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_18 (Conv2D)              │ (None, 254, 254, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_18 (MaxPooling2D) │ (None, 127, 127, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_19 (Conv2D)              │ (None, 125, 125, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_19 (MaxPooling2D) │ (None, 62, 62, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_20 (Conv2D)              │ (None, 60, 60, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_20 (MaxPooling2D) │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_21 (Conv2D)              │ (None, 28, 28, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_21 (MaxPooling2D) │ (None, 14, 14, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_22 (Conv2D)              │ (None, 12, 12, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_22 (MaxPooling2D) │ (None, 6, 6, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_23 (Conv2D)              │ (None, 4, 4, 64)       │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_23 (MaxPooling2D) │ (None, 2, 2, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_3 (Flatten)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 64)             │        16,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 183,747 (717.76 KB)

 Trainable params: 183,747 (717.76 KB)

 Non-trainable params: 0 (0.00 B)